# 用 ComfyUI 在 AMD Radeon GPU 上做文生视频

随着 AIGC（人工智能生成内容）技术的发展，文生视频、图生视频在视觉设计、艺术创作、媒体制作领域受到广泛关注。

[ComfyUI](https://github.com/comfyanonymous/ComfyUI) 是一个面向扩散模型的**节点式图形界面**，用户通过模块化的节点可视化地搭建 AI 图像/视频生成工作流。它模块化、高效、兼容性好、工作流可复用，非常适合媒体创作者提升生产力。

本教程带你在 AMD Radeon™ GPU（基于 ROCm™ 软件栈）上配置环境、安装 ComfyUI，并完成一次从文字到视频的生成。

**说明**：AMD 针对 Instinct™ GPU 提供了 ComfyUI 的 ROCm 专用实现，详见 [ComfyUI on ROCm 文档](https://rocm.docs.amd.com/projects/comfyui/en/latest/index.html)。本教程面向**消费级 / 工作站级 Radeon**（如 RX 7900 XTX）。

## 你会学到

学完本教程，你将能够：

1. 在 AMD Radeon GPU（ROCm 软件栈）上确认并准备文生视频的运行环境。
2. 安装 ComfyUI 并下载 LTX-Video 模型，搭建一条完整的文生视频工作流。
3. 用一句文字提示词生成视频，并掌握 AMD Radeon 上的关键避坑要点（如 MIOpen 导致的系统锁死）。

## 前置条件

本教程在以下环境开发并验证。

### 操作系统

- **Ubuntu 24.04.4** 或 **Ubuntu 22.04.5**。Radeon 上的 ROCm 7.2.x 目前仅支持 Ubuntu 24.04.4、Ubuntu 22.04.5、RHEL 10.1、RHEL 9.7。

### 硬件

- **AMD Radeon GPU**：本教程在 **AMD Radeon RX 7900 XTX**（RDNA3，gfx1100，24GB 显存）上测试。其他受支持型号包括 RX 7900 XT / GRE、RX 9070 XT / GRE / 9070、RX 9060 XT、Radeon AI PRO R9700、PRO W7900 / W7800 等。请确认你的卡在 [ROCm Radeon 兼容性矩阵](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/compatibility/compatibilityrad/native_linux/native_linux_compatibility.html) 内。

### 软件

- **ROCm 7.2.1**：按 [ROCm 安装指南](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) 安装后，用下面的单元验证。

In [ ]:
!amd-smi

![amd-smi 输出](../assets/amd-smi.png)

## 一、准备推理环境

> **是否需要做一、二节？** 这两节是**手动搭建运行环境**（创建 Python 环境 + 安装 ROCm 版 PyTorch）。如果你是在 Radeon cloud 提供的实例 / Docker 镜像里运行本教程，环境通常已经预装好（ROCm + PyTorch），**可直接跳到「三、验证 PyTorch」**确认环境即可，无需执行一、二节。只有从零在裸机/自建环境搭建时才需要做这两节。

下面用 `conda` 或 `venv` 创建并激活一个 Python 3.12 虚拟环境。

> 这些命令需要在**终端**里执行（而不是 notebook 单元里），因为它们要在 Jupyter 内核启动之前完成。

**用 conda**

```bash
conda create -n comfyui_env python=3.12 -y
conda activate comfyui_env
```

**或用 venv**

```bash
python3.12 -m venv comfyui_env
source comfyui_env/bin/activate
```

## 二、安装 PyTorch（ROCm 7.2.1）

在虚拟环境里安装 ROCm 版 PyTorch wheel。**AMD 官方建议用 [repo.radeon.com](https://repo.radeon.com) 上的 wheel，而不是 pytorch.org 的版本**（后者由社区滚动构建，AMD 未做充分测试）。

下面以 **Ubuntu 24.04 + Python 3.12 + ROCm 7.2.1** 为例（torch 2.9.1）。在终端执行：

```bash
# 先卸载可能存在的旧版，避免冲突
pip3 uninstall -y torch torchvision torchaudio triton

# 下载 ROCm 7.2.1 官方 wheel
BASE=https://repo.radeon.com/rocm/manylinux/rocm-rel-7.2.1
wget $BASE/torch-2.9.1%2Brocm7.2.1.lw.gitff65f5bc-cp312-cp312-linux_x86_64.whl
wget $BASE/torchvision-0.24.0%2Brocm7.2.1.gitb919bd0c-cp312-cp312-linux_x86_64.whl
wget $BASE/torchaudio-2.9.0%2Brocm7.2.1.gite3c6ee2b-cp312-cp312-linux_x86_64.whl
wget $BASE/triton-3.5.1%2Brocm7.2.1.gita272dfa8-cp312-cp312-linux_x86_64.whl

pip3 install \
  torch-2.9.1+rocm7.2.1.lw.gitff65f5bc-cp312-cp312-linux_x86_64.whl \
  torchvision-0.24.0+rocm7.2.1.gitb919bd0c-cp312-cp312-linux_x86_64.whl \
  torchaudio-2.9.0+rocm7.2.1.gite3c6ee2b-cp312-cp312-linux_x86_64.whl \
  triton-3.5.1+rocm7.2.1.gita272dfa8-cp312-cp312-linux_x86_64.whl
```

> wheel 文件名里的 git 哈希会随 ROCm 小版本变化，请以 [Install PyTorch for ROCm（Radeon）](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/native_linux/install-pytorch.html) 页面给出的当前文件名为准。Ubuntu 22.04 对应 `cp310`（Python 3.10）。

## 三、验证 PyTorch 安装

依次运行下面三个单元，确认 PyTorch 能识别到 GPU。

> **注意**：ROCm 上的 PyTorch 复用 CUDA 的 API 命名，因此 `torch.cuda.is_available()`、`torch.cuda.get_device_name()` 在 AMD GPU 上同样适用（底层走 HIP）。

In [ ]:
!python3 -c 'import torch' 2> /dev/null && echo 'Success' || echo 'Failure'

In [ ]:
!python3 -c 'import torch; print(torch.cuda.is_available())'

In [ ]:
!python3 -c "import torch; print('device name [0]:', torch.cuda.get_device_name(0))"

![PyTorch 验证输出](../assets/verify-pytorch.png)

## 四、安装 ComfyUI

从源码安装最新版 ComfyUI。

In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git

![克隆 ComfyUI](../assets/comfyui-clone.png)

In [ ]:
%cd ComfyUI
!sed -i.bak -E '/^(torch|torchaudio|torchvision)([<>=~!0-9.]*)?$/s/^/# /' requirements.txt

![注释 requirements 中的 torch](../assets/comfyui-requirements-edit.png)

In [ ]:
!pip3 install -r requirements.txt

![安装 ComfyUI 依赖](../assets/pip-install.png)

## 五、准备模型

LTX-Video 是 Lightricks 出品的高效视频模型，本教程使用 2B 蒸馏版 `ltxv-2b-0.9.6-distilled`。

需要下载两个文件：

- 视频模型 `ltxv-2b-0.9.6-distilled-04-25.safetensors` → 放到 `models/checkpoints/`
- 文本编码器 `t5xxl_fp16.safetensors` → 放到 `models/text_encoders/`

下面用 **[ModelScope（魔搭社区）](https://modelscope.cn)** 下载，国内访问速度快、稳定。

In [ ]:
!pip install -q modelscope

from modelscope import snapshot_download

snapshot_download(
    "AI-ModelScope/LTX-Video",
    allow_patterns="ltxv-2b-0.9.6-distilled-04-25.safetensors",
    local_dir="models/checkpoints",
)

snapshot_download(
    "AI-ModelScope/flux_text_encoders",
    allow_patterns="t5xxl_fp16.safetensors",
    local_dir="models/text_encoders",
)

print("模型下载完成")

![模型下载完成](../assets/models-download.png)

## ⚠️ AMD Radeon 用户重要提示

在 **AMD GPU + Linux** 上跑 ComfyUI，默认的 `--use-pytorch-cross-attention` 会触发 MIOpen 卷积内核。这些内核在部分 RDNA 卡（7800 XT、7900 XTX 等）上可能引发 `hipErrorLaunchFailure`，进而**锁死整个操作系统**（不只是进程崩溃）。该问题在 ComfyUI v0.15–v0.18 被多人报告，涉及视频生成和普通图像生成。

**推荐启动参数**：

```bash
python3 main.py --use-quad-cross-attention --disable-smart-memory
```

- `--use-quad-cross-attention`：完全绕开 MIOpen 路径（比禁用 MIOpen 大约快 20%）。
- `--disable-smart-memory`：保证显存不足时能正常报 OOM，而不是直接锁死系统。

其他可选规避方式：`export COMFYUI_ENABLE_MIOPEN=0`（彻底关掉 MIOpen，但更慢），或 `export MIOPEN_FIND_MODE=FAST`（只用系统数据库，避免构建用户数据库时卡死）。

## 六、启动 ComfyUI 服务

下面这个单元会**前台阻塞**运行 ComfyUI 服务（一直占用该单元，直到你手动停止内核）。建议把它当作服务进程，然后在浏览器里打开 GUI 操作。

In [ ]:
!python3 main.py --use-quad-cross-attention --disable-smart-memory

![ComfyUI 启动日志](../assets/comfyui-launch-log.png)

![ComfyUI 默认工作流](../assets/comfyui-default-json.png)

## 七、加载工作流

ComfyUI 的工作流定义了生成图像/视频的完整流水线与参数，格式是 JSON 文件，或编码进 WebP 动图（`*.webp`）。你可以从零搭，也可以加载现成的。

从 [ComfyUI LTXV 示例页](https://comfyanonymous.github.io/ComfyUI_examples/ltxv/) 下载 LTX-Video 文生视频工作流（`*.json` 或 `*.webp`），然后拖进 ComfyUI 界面即可加载。

> 我们用的是 **0.9.6 distilled** 模型，请选页面上**蒸馏版（distilled）**的工作流：它步数少（通常 8 步左右）、不需要 STG/CFG，速度最快。

工作流由多个节点组成，关键节点：

- **`Load Checkpoint`**：加载扩散模型。把 `ckpt_name` 选成 `ltxv-2b-0.9.6-distilled-04-25.safetensors`。
- **`Load CLIP` / 文本编码器**：加载 `t5xxl_fp16.safetensors`。
- **`CLIP Text Encode (Positive Prompt)`**：正向提示词。可改成你自己的描述。推荐示例：
  > A drone quickly rises through a bank of morning fog, revealing a pristine alpine lake surrounded by snow-capped mountains. The camera glides forward over the glassy water, capturing perfect reflections of the peaks. As it continues, the perspective shifts to reveal a lone wooden cabin with a curl of smoke from its chimney, nestled among tall pines at the lake's edge. The final shot tracks upward rapidly, transitioning from intimate to epic as the full mountain range comes into view, bathed in the golden light of sunrise breaking through scattered clouds.
  > （提示词建议用英文，模型对英文理解更好。）
- **`EmptyLTXVLatentVideo`**：控制生成视频的分辨率（宽高）与帧数。显存吃紧时先调小分辨率/帧数。

更多节点说明见 [ComfyUI wiki](https://comfyui-wiki.com/en/comfyui-nodes)。

![加载并设置好的 LTX 工作流](../assets/comfyui-ltx-workflow.png)

## 八、运行推理

工作流准备好后，点击界面上的 **Run / Queue Prompt** 启动整条流水线。终端日志大致如下：

```
got prompt
model weight dtype torch.bfloat16, manual cast: None
Requested to load MochiTEModel_
Requested to load LTXV
100%|██████████████████████████████| 8/8 [00:xx<00:00,  x.xxit/s]
Requested to load VideoVAE
Prompt executed in xx.xx seconds
```

最终视频显示在 `SaveAnimatedWEBP` 节点里，`*.webp` 文件保存在 `ComfyUI/output/` 目录下。

> **若遇到 VAE 解码 OOM**：日志出现 `Ran out of memory when regular VAE decoding, retrying with tiled VAE decoding` 是正常回退，ComfyUI 会自动改用分块（tiled）VAE 解码。若仍 OOM，手动加一个 VAE tiler 节点（如 512×512 tile），或调小 `EmptyLTXVLatentVideo` 的分辨率与帧数。

![生成的视频效果](../assets/comfyui-output.webp)

*生成的视频效果（LTX-Video 文生视频，晨雾雪山高山湖泊无人机镜头）。动图来自 AMD 原版 ComfyUI 文生视频教程，作效果示意。*

## 进阶：升级到更高质量的模型（可选）

本教程主路径用的是轻量快速的 **LTXV 2B 0.9.6 distilled**。如果你想要更高画质，可按显存选择以下升级，工作流和模型文件都不同：

| 模型 | 说明 | 显存建议 | 工作流 |
| --- | --- | --- | --- |
| `ltxv-2b-0.9.6-distilled`（本教程） | 2B 蒸馏，最快、显存最省 | 8–24GB | ComfyUI 内置 LTXV distilled 示例 |
| `ltxv-13b-0.9.8-distilled` / `-fp8` | 13B 蒸馏，画质更好，支持最长 60 秒 | 24GB（fp8 更省） | [ComfyUI-LTXVideo](https://github.com/Lightricks/ComfyUI-LTXVideo) 提供的 13B 工作流 |
| `ltxv-13b-0.9.8-dev` / `-fp8` | 13B 全量，画质最高 | 24GB+（建议 fp8） | 同上 |
| **LTX-2**（最新一代） | 已内置进 ComfyUI 核心，能力最强但模型偏大（22B） | 需较大显存，消费级 Radeon 谨慎评估 | [LTX-2 官方工作流](https://github.com/Lightricks/ComfyUI-LTXVideo) |

13B / LTX-2 的工作流通常需要单独加载 VAE、文本编码器和升采样器（不是单文件 checkpoint），请安装 ComfyUI Manager 后通过 [ComfyUI-LTXVideo](https://github.com/Lightricks/ComfyUI-LTXVideo) 节点包加载对应工作流，并按其 README 下载所需模型文件。

## 课后测验

**第 1 题：** 在 **ROCm 7.x** 上查看 AMD GPU 状态与 ROCm 版本，应使用下面哪个命令？

- A. `nvidia-smi`
- B. `rocm-smi`
- C. `amd-smi`
- D. `lspci -k`

**第 2 题：** 在 **AMD Radeon + Linux** 上用 ComfyUI 做图像/视频生成，为避免 MIOpen 卷积内核触发 `hipErrorLaunchFailure`、进而锁死整个系统，推荐的启动参数是？

- A. `--use-pytorch-cross-attention`（默认值）
- B. `--use-quad-cross-attention --disable-smart-memory`
- C. `--cpu`
- D. `--highvram`

**第 3 题：** 在 Radeon 上安装 PyTorch 时，AMD 官方推荐从哪里获取 wheel？

- A. PyPI 默认的 `pip install torch`（CUDA 版）
- B. AMD 官方仓库 `repo.radeon.com` 的 ROCm 专用 wheel
- C. conda 的 nvidia 频道
- D. 必须自己从源码编译

<details>
<summary>点击查看答案与解析</summary>

1. **C**。ROCm 6.4 及更早用 `rocm-smi`；**ROCm 7.x 起统一用 `amd-smi`**。`nvidia-smi` 是 NVIDIA 的工具，不适用于 AMD GPU。
2. **B**。`--use-quad-cross-attention` 完全绕开 MIOpen 路径，`--disable-smart-memory` 保证显存不足时正常报 OOM 而非锁死系统；而默认的 `--use-pytorch-cross-attention` 恰恰是触发问题的那个。
3. **B**。AMD 官方建议用 `repo.radeon.com` 上的 ROCm 专用 wheel（如 `torch-2.9.1+rocm7.2.1...`）；PyPI 默认装的是 CUDA 版，在 AMD GPU 上无法使用。

</details>